# 1 · MLflow chunking sweep with RAGAS faithfulness per config (+ Registry)

**Kaggle · GPU T4 x2 · Internet ON.** Runs the 5 chunking configs (chunk_size 256/384/512/768 + window baseline) on an even
20-question subset with Qwen3-8B (vLLM) as generator *and* RAGAS judge, logs each as an MLflow run, registers the best config in the
Model Registry as `civil-code-rag-chunking` (alias `production` + stage Production) and writes `reports/ragas_ci.json` (the file the CI gate reads).
Before running: make sure `data/evaluation/rag_eval.json` is committed to the repo (it is, in this project). Budget ~2 GPU-hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/AhmedYasser06/Egyptian-Civil-Code-RAG.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

## Corpus
`dvc pull` fetches the PDF, the JSON corpus, the chunks and the embeddings from the shared DVC remote.
If the remote needs credentials, add Kaggle secrets `DVC_USER` / `DVC_PASSWORD` (Add-ons -> Secrets).
Fallback: attach a Kaggle dataset containing your `data/processed/` folder.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
try:
    s = UserSecretsClient()
    user, pw = s.get_secret("DVC_USER"), s.get_secret("DVC_PASSWORD")
    !pip install -q "dvc==3.67.1" "dvc-s3==3.3.0"
    !dvc remote modify --local origin auth basic
    !dvc remote modify --local origin user {user}
    !dvc remote modify --local origin password {pw}
except Exception as e:
    print("no DVC secrets, trying anonymous pull:", e)
    !pip install -q "dvc==3.67.1" "dvc-s3==3.3.0"
!dvc pull
!ls -la data/processed
# fallback if dvc pull failed:  !mkdir -p data/processed && cp /kaggle/input/*/final-*.json /kaggle/input/*/embedded-*.json data/processed/

## vLLM in its own environment
vLLM pins its own torch/transformers, so it gets a separate venv (made with `uv`: Kaggle's `python -m venv` has no ensurepip). Kaggle doesn't allow `!cmd &`, so servers start with `subprocess.Popen`.

In [ ]:
!pip install -q uv
!uv venv /kaggle/vllm-env --python 3.11
!uv pip install -q --python /kaggle/vllm-env/bin/python vllm wrapt
import os, signal, subprocess, time, urllib.request

def start(cmd, log_path, env=None):
    return subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **(env or {})}, start_new_session=True)

def wait_ready(url, log_path, minutes=30):
    for i in range(minutes * 4):
        try:
            urllib.request.urlopen(url, timeout=5); print(f"ready after ~{i*15}s"); return True
        except Exception:
            time.sleep(15)
    print("NOT ready -- log:"); os.system(f"tail -n 30 {log_path}"); return False

def stop(proc):
    os.killpg(proc.pid, signal.SIGTERM)
    try: proc.wait(timeout=120)
    except subprocess.TimeoutExpired: os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(10)

def start_vllm(model, log_path, util=0.75):
    return start(["/kaggle/vllm-env/bin/vllm", "serve", model, "--tensor-parallel-size", "2",
                  "--dtype", "half", "--max-model-len", "8192", "--gpu-memory-utilization", str(util),
                  "--port", "8000"], log_path, env={"PYTHONPATH": ""})

In [ ]:
# Project dependencies (RAGAS pins openai<2 through instructor, so keep the project's pins)
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
!mkdir -p eval_out reports models

## Serve Qwen3-8B (fp16) with vLLM

In [ ]:
vllm = start_vllm("Qwen/Qwen3-8B", "/kaggle/working/vllm.log")
wait_ready("http://localhost:8000/v1/models", "/kaggle/working/vllm.log")

## Sweep (resumable: finished configs are cached in `eval_out/sweep/`)

In [ ]:
os.environ.update(LLM_PROVIDER="VLLM", VLLM_BASE_URL="http://localhost:8000/v1", VLLM_MODEL_ID="Qwen/Qwen3-8B",
                  JUDGE_PROVIDER="vllm", JUDGE_BASE_URL="http://localhost:8000/v1", JUDGE_API_KEY="EMPTY", JUDGE_MODEL="Qwen/Qwen3-8B",
                  JUDGE_MAX_TOKENS="4096", JUDGE_EXTRA_BODY='{"chat_template_kwargs":{"enable_thinking":false}}',
                  MLFLOW_TRACKING_URI="sqlite:////kaggle/working/repo/mlflow.db", MLFLOW_DISABLE_AGENT_HINT="1")
!python -m src.mlflow.chunking_ragas_sweep --questions data/evaluation/rag_eval.json --n 20 --device cuda:1 > eval_out/sweep.log 2>&1; tail -n 15 eval_out/sweep.log

In [ ]:
import mlflow, pandas as pd
mlflow.set_tracking_uri("sqlite:////kaggle/working/repo/mlflow.db")
runs = mlflow.search_runs(experiment_names=["civil-code-rag-chunking"])
cols = ["tags.mlflow.runName","params.chunk_size","params.overlap","params.embedding_model","metrics.faithfulness",
        "metrics.answer_relevancy","metrics.context_precision","metrics.context_recall","metrics.article_hit_rate"]
display(runs[[c for c in cols if c in runs.columns]].sort_values("metrics.faithfulness", ascending=False))
!python scripts/ragas_gate.py

In [ ]:
!zip -qr /kaggle/working/sweep_results.zip mlflow.db mlruns reports eval_out/sweep eval_out/sweep.log
from IPython.display import FileLink
display(FileLink("/kaggle/working/sweep_results.zip"))